In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vedshivhare/combined-platform-courses-cleaned/online_courses_cleaned.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/vedshivhare/combined-platform-courses-cleaned/online_courses_cleaned.csv")

In [3]:
df.head()

,course_title,rating,reviewcount,level,Course_Id,description,duration_hours,platform
0,Financial Mathematics - Theory of Interest & C...,4.3,341.0,All Levels,15128,By MJ the Fellow Actuary,3.0,Udemy
1,C Programming Masterclass: Pointers & Advanced...,4.6,804.0,Intermediate,3028,"C Programming Advanced Topics: Pointers, Memor...",41.0,Udemy
2,Mastering QuickBooks 2015 Made Easy Training T...,3.7,32.0,All Levels,17066,Learn Introductory through Advanced material w...,13.5,Udemy
3,Documentary Filmmaking Step by Step,4.2,750.0,All Levels,25803,"From a 3-time Sundance alum, learn to make doc...",4.5,Udemy
4,Ultra-Fast WordPress Speed With 10Web WordPres...,4.6,42.0,Beginner,10128,Boost WordPress SEO with ultra-fast WordPress ...,1.5,Udemy


In [4]:
df['course_title'].unique()

array(['Financial Mathematics - Theory of Interest & Cashflow Models',
       'C Programming Masterclass: Pointers & Advanced C Language',
       'Mastering QuickBooks 2015 Made Easy Training Tutorial', ...,
       'Digital Marketing & Performance Marketing fundamental course',
       'Value Investing: How to Invest Wisely Like Warren Buffett',
       'The Digital Yuan: The World’s Most Advanced CBDC'], dtype=object)

In [5]:
df['description'].unique()

array(['By MJ the Fellow Actuary',
       'C Programming Advanced Topics: Pointers, Memory, Low-Level C Language and Embedded C Preparation',
       'Learn Introductory through Advanced material with this complete QuickBooks course. Video lessons & manuals included. ',
       ..., 'From creative coding basics to advanced algorithmic art',
       'The systematic course to get you started in learn digital marketing & performance marketing (includes Google Ads course)',
       'Value Investing Made Simple - Proven & Concise Investment Guide for Beginners to Start Investing in The Stock Market'],
      dtype=object)

In [6]:
# Function to balance the weights
def balance_metadata(row):
    title = str(row['course_title'])
    desc = str(row['description'])
    
    if row['platform'] == 'Coursera':
        # Description already contains the title once. 
        # Adding title + description = Title x2
        return title + " " + desc
    else:
        # Udemy description does NOT contain the title.
        # Adding title + title + description = Title x2
        return title + " " + title + " " + desc

# Apply the balanced logic
df['metadata_balanced'] = df.apply(balance_metadata, axis=1)

In [7]:
import re

def clean_for_nlp(text):
    # 1. Convert to lowercase
    text = str(text).lower()
    
    # 2. Remove special characters and punctuation
    # [^a-zA-Z0-9\s] means: "Match anything that is NOT a letter, number, or space"
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # 3. Remove extra whitespace (replace multiple spaces with one)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply this to your balanced metadata column
df['metadata'] = df['metadata_balanced'].apply(clean_for_nlp)

In [8]:
print("Coursera Sample:")
print(df[df['platform'] == 'Coursera']['metadata'].iloc[2])

print("\nUdemy Sample:")
print(df[df['platform'] == 'Udemy']['metadata'].iloc[2])

Coursera Sample:
viral marketing and how to craft contagious content viral marketing and how to craft contagious content brand management communication critical thinking influencing marketing media strategy planning people analysis social media strategy market analysis

Udemy Sample:
mastering quickbooks 2015 made easy training tutorial mastering quickbooks 2015 made easy training tutorial learn introductory through advanced material with this complete quickbooks course video lessons manuals included


In [9]:
df.drop(['description' , 'metadata_balanced'],axis = 1, inplace = True)

In [10]:
df.describe()

,rating,reviewcount,Course_Id,duration_hours
count,27028.000000,27028.000000,27028.000000,27028.000000
mean,4.308425,1777.716109,13514.500000,16.290638
std,0.523703,9133.326013,7802.455874,62.816272
min,0.000000,0.000000,1.000000,0.000000
25%,4.100000,81.000000,6757.750000,2.000000
50%,4.400000,300.000000,13514.500000,4.500000
75%,4.600000,936.000000,20271.250000,10.000000
max,5.000000,486391.000000,27028.000000,2520.000000


In [11]:
# C is the average rating across all 27,000 courses
C = df['rating'].mean()

# m is the minimum reviews required to be considered "reliable"
# Using the 75th percentile means a course needs more reviews than 75% of the data to be fully 'trusted'
m = df['reviewcount'].quantile(0.75)

print(f"Global Average Rating (C): {C:.2f}")
print(f"Minimum Review Threshold (m): {m:.0f}")

def calculate_weighted_score(row):
    v = row['reviewcount']
    R = row['rating']
    # The formula: (v/(v+m) * R) + (m/(v+m) * C)
    return (v / (v + m) * R) + (m / (v + m) * C)

df['weighted_score'] = df.apply(calculate_weighted_score, axis=1)

Global Average Rating (C): 4.31
Minimum Review Threshold (m): 936


In [12]:
df.head()

,course_title,rating,reviewcount,level,Course_Id,duration_hours,platform,metadata,weighted_score
0,Financial Mathematics - Theory of Interest & C...,4.3,341.0,All Levels,15128,3.0,Udemy,financial mathematics theory of interest cashf...,4.306175
1,C Programming Masterclass: Pointers & Advanced...,4.6,804.0,Intermediate,3028,41.0,Udemy,c programming masterclass pointers advanced c ...,4.443153
2,Mastering QuickBooks 2015 Made Easy Training T...,3.7,32.0,All Levels,17066,13.5,Udemy,mastering quickbooks 2015 made easy training t...,4.288311
3,Documentary Filmmaking Step by Step,4.2,750.0,All Levels,25803,4.5,Udemy,documentary filmmaking step by step documentar...,4.260193
4,Ultra-Fast WordPress Speed With 10Web WordPres...,4.6,42.0,Beginner,10128,1.5,Udemy,ultra fast wordpress speed with 10web wordpres...,4.320946


In [13]:
df['platform'].value_counts()

platform
Udemy       25889
Coursera     1139
Name: count, dtype: int64

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the Vectorizer
# stop_words='english' removes common words like 'the', 'is', 'and'
# max_features=10000 keeps the most important 10,000 words to save memory
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=10000)

# 2. Fit and Transform the metadata
# Use .fillna('') to ensure the model doesn't crash on empty strings
tfidf_matrix = tfidf.fit_transform(df['metadata'].fillna(''))

# 3. Check the shape
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
# You should see (27028, 10000)

TF-IDF Matrix Shape: (27028, 10000)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def get_recommendations(user_query, top_n=50):
    # 1. Convert the user's query into the same TF-IDF vector format
    query_vec = tfidf.transform([user_query.lower()])
    
    # 2. Calculate Similarity between the query and all 27k courses
    # similarity_scores will be an array of numbers between 0 and 1
    similarity_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 3. Add the scores to your dataframe
    df_temp = df.copy()
    df_temp['similarity_score'] = similarity_scores
    
    return df_temp

In [16]:
def recommend_best_learning(query, level_filter=None, max_hours=None):
    # Get the similarity scores
    results = get_recommendations(query)
    
    # Apply Student Filters
    if level_filter:
        results = results[results['level'] == level_filter]
    
    if max_hours:
        results = results[results['duration_hours'] <= max_hours]
        
    # FINAL STEP: Rank by Similarity (primary) and Weighted Score (secondary)
    # This ensures a highly relevant course that is ALSO highly trusted comes first
    final_selection = results.sort_values(
        by=['similarity_score', 'weighted_score'], 
        ascending=False
    ).head(5)
    
    return final_selection[['course_title', 'reviewcount' , 'platform', 'level', 'duration_hours', 'rating', 'weighted_score']]

In [17]:
# Test Case: Finance & Math
recommend_best_learning("Data Science and Machine Learning", max_hours=20)

,course_title,reviewcount,platform,level,duration_hours,rating,weighted_score
24029,Mastering Data Science and Machine Learning Fu...,1397.0,Udemy,All Levels,2.0,4.5,4.423140
19385,Data Science and Machine Learning Bootcamp with R,16468.0,Udemy,All Levels,18.0,4.7,4.678941
7361,Data Science A-Z : Machine Learning with Pytho...,176.0,Udemy,All Levels,12.5,4.1,4.275437
11265,Excel for Data Science and Machine Learning,429.0,Udemy,All Levels,6.0,4.5,4.368634
13898,"Data science, machine learning, and analytics ...",44.0,Udemy,Beginner,3.0,4.6,4.321516


In [18]:
# Test Case: Quick Excel Skills
recommend_best_learning("Excel Pivot Tables", max_hours=2)

,course_title,reviewcount,platform,level,duration_hours,rating,weighted_score
18215,Excel Pivot Tables in a Nutshell,918.0,Udemy,Beginner,0.516667,4.2,4.254739
19294,Learn Excel Pivot Tables in 2 Hours with Dr. J...,117.0,Udemy,Beginner,2.000000,4.9,4.374155
5193,No Nonsense Excel Pivot Tables Pro Course,1876.0,Udemy,Advanced,1.000000,4.8,4.636375
5723,Beginners Guide to Microsoft Excel Pivot Tables,2042.0,Udemy,Intermediate,0.550000,4.7,4.576926
1614,Master Excel Pivot Tables - Excel 365 and Exce...,12869.0,Udemy,All Levels,1.500000,4.5,4.487011


In [19]:
# Searching for Professional Certifications
recommend_best_learning("IBM Data Science Professional Certificate", max_hours=100)

,course_title,reviewcount,platform,level,duration_hours,rating,weighted_score
11733,IBM Data Science,0.0,Coursera,Beginner,0.0,0.0,4.308425
15258,IBM Data Science,0.0,Coursera,Beginner,0.0,0.0,4.308425
2008,IBM Data Analyst,0.0,Coursera,Beginner,0.0,0.0,4.308425
25025,IBM Data Analyst,0.0,Coursera,Beginner,0.0,0.0,4.308425
26042,Be Aware of Data Science,49.0,Udemy,Beginner,6.0,4.7,4.327904
